<a href="https://colab.research.google.com/github/mariakellyeduarda17-ops/UFV/blob/main/1704.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install PySUS pandas

In [7]:
import pandas as pd
from pysus.online_data import SIH

def extrair_dados_itu():
    # Lista de códigos IBGE 6 dígitos (MG > 100k hab)
    cidades = [
        '310620', '317020', '311860', '313670', '310670', '314330',
        '315460', '317010', '312770', '313130', '315780', '316860',
        '312230', '315210', '310560', '313820', '313170', '310160',
        '314710', '314790', '311830', '316720', '310350'
    ]

    # Parâmetros
    ano = 2024
    meses = [1, 2] # Teste com 2 meses
    resultados = []

    for m in meses:
        print(f"Baixando mes {m}...")
        try:
            # Download
            data = SIH.download(states='MG', years=ano, months=m, groups='RD')
            df = data.to_dataframe()

            # Limpeza de dados
            df['DIAG_PRINC'] = df['DIAG_PRINC'].str.strip()
            df['MUNIC_RES'] = df['MUNIC_RES'].astype(str).str.strip()

            # Filtros simplificados para evitar erro de aspas
            cid_alvo = 'N390'
            df_filtro = df[df['DIAG_PRINC'] == cid_alvo].copy()
            df_final = df_filtro[df_filtro['MUNIC_RES'].isin(cidades)].copy()

            if not df_final.empty:
                res = df_final.groupby('MUNIC_RES').size().reset_index(name='Qtd')
                res['Mes'] = m
                resultados.append(res)

        except Exception as e:
            print(f"Erro: {e}")

    if resultados:
        consolidado = pd.concat(resultados, ignore_index=True)
        return consolidado
    return pd.DataFrame()

# Rodar a função
df_total = extrair_dados_itu()
print(df_total)

Baixando mes 1...


RDMG2401.parquet: 100%|██████████| 429k/429k [00:24<00:00, 17.9kB/s]


Baixando mes 2...


RDMG2402.parquet: 100%|██████████| 430k/430k [00:29<00:00, 14.5kB/s]


   MUNIC_RES  Qtd  Mes
0     310160   11    1
1     310350   27    1
2     310560    9    1
3     310620  250    1
4     310670   20    1
5     311830   12    1
6     311860   27    1
7     312230    8    1
8     312770   13    1
9     313130   27    1
10    313170   12    1
11    313670   56    1
12    313820    4    1
13    314330   40    1
14    314710    9    1
15    314790   13    1
16    315210   10    1
17    315460   32    1
18    315780   56    1
19    316720    5    1
20    316860    5    1
21    317010   38    1
22    317020  133    1
23    310160    3    2
24    310350   46    2
25    310560    6    2
26    310620  161    2
27    310670   12    2
28    311830    9    2
29    311860   17    2
30    312230    9    2
31    312770   29    2
32    313130   33    2
33    313170   11    2
34    313670   49    2
35    313820    9    2
36    314330   40    2
37    314710    7    2
38    314790    6    2
39    315210   13    2
40    315460   40    2
41    315780   38    2
42    31672

In [8]:
# Dicionário para traduzir os códigos que você já extraiu
nomes_cidades = {
    '310620': 'Belo Horizonte', '317020': 'Uberlândia', '311860': 'Contagem',
    '313670': 'Juiz de Fora', '310670': 'Betim', '314330': 'Montes Claros',
    '315460': 'Ribeirão das Neves', '317010': 'Uberaba', '312770': 'Governador Valadares',
    '313130': 'Ipatinga', '315780': 'Santa Luzia', '316860': 'Teófilo Otoni',
    '312230': 'Divinópolis', '315210': 'Pouso Alegre', '310560': 'Barbacena',
    '313820': 'Lavras', '313170': 'Itabira', '310160': 'Alfenas',
    '314710': 'Pará de Minas', '314790': 'Passos', '311830': 'Conselheiro Lafaiete',
    '316720': 'Sete Lagoas', '310350': 'Araguari'
}

# 1. Adiciona os nomes das cidades
df_total['Nome_Cidade'] = df_total['MUNIC_RES'].map(nomes_cidades)

# 2. Reorganiza as colunas para ficar mais legível
df_final = df_total[['Mes', 'Nome_Cidade', 'Qtd']].sort_values(['Mes', 'Qtd'], ascending=[True, False])

print("--- TOP 10 INTERNAÇÕES POR ITU (JAN/FEV 2024) ---")
print(df_final.head(10).to_string(index=False))

# 3. Opcional: Salvar em Excel ou CSV
# df_final.to_csv('internacoes_itu_mg_2024.csv', index=False, encoding='utf-8-sig')

--- TOP 10 INTERNAÇÕES POR ITU (JAN/FEV 2024) ---
 Mes        Nome_Cidade  Qtd
   1     Belo Horizonte  250
   1         Uberlândia  133
   1       Juiz de Fora   56
   1        Santa Luzia   56
   1      Montes Claros   40
   1            Uberaba   38
   1 Ribeirão das Neves   32
   1           Araguari   27
   1           Contagem   27
   1           Ipatinga   27
